# Exp 2 - Latency and Jitter Measurement using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Measure latency and jitter for a packet stream and explain why timing variation matters in autonomous communication.

- **Latency:** end-to-end delay for one packet.
- **Jitter:** variation in latency across packets.
- **Tail latency:** high-percentile latency such as P95 or P99, useful for identifying rare but important timing spikes.

## Core Real-Time Systems Theory Notes

### 1. Introduction to Real-Time Systems

A real-time system is a computing system in which correctness depends on two things: the logical correctness of the output and the time at which the output is produced. In a normal general-purpose system, a late answer may be inconvenient. In a real-time system, a late answer may be useless or may cause unsafe behavior.

Logical correctness means that the calculated value or decision is correct. Temporal correctness means that the value or decision is available within the required time bound. A vehicle braking controller, robotic arm controller, industrial motor drive, medical monitoring device, avionics controller, or power-grid protection unit must satisfy both.

Real-time does not simply mean "fast." A fast system that sometimes misses its deadline is not dependable for hard real-time control. A slower system with bounded and predictable timing may be more suitable if it always meets the required deadline. The key engineering properties are determinism, predictability, bounded latency, and analyzable worst-case behavior.

General-purpose systems optimize average response, throughput, fairness, and user convenience. Real-time systems optimize deadline satisfaction, bounded response time, and predictable behavior under defined load. This is why real-time operating systems, embedded controllers, field buses, and deterministic networks often use priority policies, static configuration, time slots, or admission control.

### 2. Classification of Real-Time Systems

Hard real-time systems must not miss deadlines. A missed deadline is treated as system failure. Examples include autonomous emergency braking, airbag control, flight-control surfaces, pacemaker control, and industrial safety shutdown.

Firm real-time systems can tolerate some missed deadlines, but a late result has no value and is discarded. Examples include object-detection frames that arrive after the object is no longer relevant, traffic-sign recognition after the vehicle has passed the sign, or a stale cooperative-awareness message in V2X communication.

Soft real-time systems tolerate deadline misses with quality degradation. Examples include dashboard display refresh, infotainment audio buffering, non-critical telemetry upload, passenger comfort control, and route-estimation updates.

The classification depends on the consequence of lateness, not only the application name. A camera pipeline may be hard real-time when used for emergency braking, firm real-time when used for immediate object tracking, and soft real-time when used for driver display recording.

### 3. Real-Time Tasks and Events

A task is a schedulable unit of computation. In autonomous systems, tasks may represent sensor sampling, frame processing, message transmission, controller update, actuator command generation, logging, or security checking.

Periodic tasks occur at fixed intervals. Example: sample wheel speed every 10 ms. A periodic task is commonly described by execution time C, period T, and deadline D.

Aperiodic tasks occur irregularly and do not have a guaranteed minimum inter-arrival time. Example: a user opens a diagnostic screen. Aperiodic work is often lower criticality or handled by background servers.

Sporadic tasks occur irregularly but have a known minimum separation between arrivals. Example: emergency obstacle events may occur unpredictably but cannot arrive faster than a defined physical or system limit. Sporadic modelling is useful because it allows worst-case analysis.

Time-triggered events are released by a clock schedule. They improve predictability because activation times are known in advance. Event-triggered events are released when an external condition occurs, such as receiving a packet, detecting an obstacle, or crossing a threshold. Event-triggered systems are responsive but require careful overload handling.

### 4. Timing Parameters

The event occurrence time is the real-world time at which the physical event occurs. Release time is when the corresponding task becomes ready for scheduling. Arrival time is often used for the time at which a job enters a queue or a packet reaches a node. Start time is when execution actually begins. Execution time or computation time is the CPU or processor time consumed by the job.

Waiting time is the time spent ready but not executing:

```
waiting_time = start_time - release_time
```

Completion time or finish time is when the job finishes. Response time is the delay from release or arrival to completion:

```
response_time = finish_time - release_time
```

In many lab contexts, turnaround time is also:

```
turnaround_time = finish_time - arrival_time
```

If release time and arrival time are the same, response time and turnaround time become numerically equal. In networked systems they may differ because a real-world event can occur before the software task is released, or a packet can be generated before it reaches the receiving queue.

### 5. Timing Constraints

A relative deadline is measured from release time. An absolute deadline is a time on the system timeline:

```
absolute_deadline = release_time + relative_deadline
```

A deadline is met when:

```
finish_time <= absolute_deadline
```

A deadline miss occurs when:

```
finish_time > absolute_deadline
```

Deadline margin shows how much time remains at completion:

```
deadline_margin = absolute_deadline - finish_time
```

Positive margin means the task finished early. Zero means it finished exactly at the deadline. Negative margin means a miss.

Slack time estimates available spare time before a deadline:

```
slack = absolute_deadline - current_time - remaining_execution_time
```

Laxity is often used similarly:

```
laxity = deadline - current_time - remaining_computation_time
```

Worst-Case Execution Time, or WCET, is the maximum execution time under defined assumptions. Best-Case Execution Time, or BCET, is the minimum. Average execution time is not enough for hard real-time certification because rare long execution paths still matter.

### 6. Communication Performance Parameters

Latency is the time taken for data to move from source to destination:

```
latency = receive_time - send_time
```

Jitter is variation in latency. A simple packet-to-packet jitter estimate is:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Throughput is useful delivered data per unit time:

```
throughput = delivered_bits / observation_time
```

Bandwidth is the nominal or available capacity of a link. Throughput is what is actually achieved after overhead, contention, retransmission, protocol limits, and congestion.

Packet transmission time is:

```
transmission_time = packet_size_bits / link_rate_bits_per_second
```

End-to-end delay can be modeled as:

```
end_to_end_delay = processing_delay + queueing_delay + transmission_delay + propagation_delay
```

Communication overhead is the extra data or time consumed by headers, acknowledgements, encryption, retransmission, routing, and synchronization. Packet loss affects reliability:

```
packet_loss_rate = lost_packets / sent_packets
reliability = delivered_packets / sent_packets
```

### 7. Real-Time Communication Requirements

Bounded latency means there is a known upper limit for message delay under defined conditions. Low jitter means delay stays stable across transmissions. Predictable communication means the designer can reason about message timing before deployment. Reliability means messages are delivered with acceptable probability or with recovery mechanisms. Availability means the communication service is usable when needed.

Deterministic message delivery is often achieved through priority arbitration, time slots, traffic shaping, redundancy, admission control, or real-time Ethernet features. Deadline-aware communication means messages are scheduled according to urgency and usefulness, not simply first-come first-served.

### 8. Timing Analysis in Autonomous Systems

A typical autonomous timing chain is:

```
Sensor -> Perception -> Planning/Control -> Actuator -> Physical Response
```

The perception-to-action delay is:

```
perception_to_action_delay =
    sensor_capture_time
  + sensor_preprocessing_time
  + perception_inference_time
  + planning_time
  + control_time
  + communication_time
  + actuator_response_time
```

For an autonomous braking example:

```
stopping_distance = reaction_distance + braking_distance
reaction_distance = vehicle_speed * total_system_delay
braking_distance = vehicle_speed^2 / (2 * deceleration)
```

Deadline verification compares the computed or measured response time with the maximum safe response time:

```
system_is_timely = measured_response_time <= required_deadline
```

Case Study - Autonomous Emergency Braking:
A front sensor detects an obstacle at a fixed distance. The system must capture sensor data, process it, decide, transmit the command, and apply braking before the remaining stopping distance becomes unsafe. The case study shows why real-time correctness is a chain property. A fast perception algorithm alone is not enough if the actuator command is delayed.

Case Study - Robotic Arm in Industrial Automation:
A robotic arm must stop when a worker crosses a safety boundary. Sensor detection, controller scheduling, network delivery, and motor-drive response must all be bounded. High average throughput is irrelevant if one delayed safety packet allows the arm to continue moving too long.

Case Study - V2X Hazard Warning:
A vehicle broadcasts a hazard message to nearby vehicles. The message is useful only if received before the receiving vehicle must react. This connects communication latency, jitter, packet loss, message freshness, and security verification.

### 9. Textbook Design Workflow for Real-Time Experiments

When solving a real-time lab problem, use a disciplined workflow. First identify the physical event or communication event. Second identify the software task or network message created by that event. Third list the timing parameters: release time, start time, execution time, finish time, and deadline. Fourth compute the response time and deadline margin. Fifth classify the consequence of lateness as hard, firm, or soft. Sixth propose a design improvement if the deadline is missed.

For autonomous systems, the timing boundary should be tied to a physical reason. For example, a braking deadline should relate to speed, distance, and deceleration. A communication deadline should relate to how long a message remains useful. A security verification deadline should relate to whether authentication or IDS checks finish before the receiver uses the message.

### 10. Common Architectures Used Across These Experiments

Most experiments in this lab can be understood using one of three architecture patterns.

Control-loop pattern:

```
Sensor -> Controller Task -> Actuator -> Plant / Vehicle -> Sensor
```

Communication-loop pattern:

```
Publisher / Sender -> Network Medium -> Receiver / Subscriber -> Application Decision
```

Security-monitoring pattern:

```
Message Source -> Security Check -> IDS / Risk Logic -> Accept, Reject, or Alert
```

The control-loop pattern focuses on WCET, response time, and deadline satisfaction. The communication-loop pattern focuses on latency, jitter, throughput, packet loss, and deterministic delivery. The security-monitoring pattern focuses on integrity, authentication, replay resistance, anomaly detection, and risk reduction. Autonomous systems usually combine all three patterns, which is why timing and security cannot be treated as separate afterthoughts.

### 11. Common Mistakes to Avoid in Lab Answers

Do not say "real-time means fast." Say "real-time means deadline-bound." Do not use average execution time as a substitute for WCET in hard real-time analysis. Do not conclude that high throughput guarantees good real-time performance. Do not claim a security mechanism provides authentication unless the mechanism actually proves sender identity. Do not claim a physical simulator or broker was used if the notebook uses a Python fallback. Clear assumptions make the lab record more credible.

### Core References for These Notes

- Python timing functions such as `perf_counter()` and monotonic clocks are documented by the official Python `time` module documentation: https://docs.python.org/3/library/time.html
- IEEE 802.1 Time-Sensitive Networking is the IEEE working-group area for time-sensitive network behavior: https://1.ieee802.org/tsn/
- SUMO official documentation describes traffic simulation concepts used in V2V mobility experiments: https://sumo.dlr.de/docs/
- MQTT is an OASIS publish-subscribe messaging standard for IoT telemetry: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- NIST FIPS 180-4 specifies SHA-256 as part of the Secure Hash Standard: https://csrc.nist.gov/pubs/fips/180-4/upd1/final
- NIST SP 800-30 Rev. 1 provides risk-assessment guidance: https://csrc.nist.gov/pubs/sp/800/30/r1/final

## Extended Experiment Notes and Case Studies

### Experiment Focus

This experiment measures latency and jitter from repeated message timing samples. In real-time networks, stable delay is often more important than the lowest average delay. A system with low mean latency but large jitter can destabilize control loops.

### Experiment Architecture

```
Message Generator
  -> Send Timestamp
  -> Simulated Network Delay
  -> Receive Timestamp
  -> Latency Series
  -> Jitter and Deadline Analysis
```

Each sample represents one message or packet. The send and receive timestamps allow latency calculation. A sequence of latencies allows jitter calculation.

### Detailed Formula Set

```
latency_i = receive_time_i - send_time_i
mean_latency = sum(latency_i) / n
max_latency = max(latency_i)
jitter_i = abs(latency_i - latency_(i-1))
mean_jitter = sum(jitter_i) / (n - 1)
deadline_violation_rate = late_packets / total_packets
```

For real-time communication, maximum latency and high-percentile latency are more meaningful than mean latency alone.

### Case Study 1 - Cooperative Adaptive Cruise Control

Vehicles exchange speed and acceleration messages. Low jitter allows the following vehicle to update smoothly. High jitter can cause uneven control commands and unsafe spacing.

### Case Study 2 - Remote Vehicle Operation

A remote operator sends steering commands over a network. A single large delay spike during a turn may be more important than many normal packets. This is why the lab must report maximum latency and jitter.

### Case Study 3 - Industrial Robot Cell

A controller sends synchronized commands to multiple actuators. Jitter can cause actuators to move out of coordination. The system may need time-triggered communication or synchronized clocks.

### Lab Record Guidance

Include a latency table, average latency, maximum latency, jitter values, and deadline violations. Do not write "network is good" from average latency alone.

## Architecture

```text
Packet Generator
  |-- packet id
  |-- send timestamp
          |
          v
Network Delay Model
  |-- base delay
  |-- random variation
  |-- occasional burst delay
          |
          v
Receiver
  |-- receive timestamp
          |
          v
Timing Analyzer
  |-- latency per packet
  |-- jitter between adjacent packets
  |-- mean, standard deviation, P95, P99
```

The model is deterministic because it uses fixed random seeds. That makes the result reproducible across notebook runs.

## Formulas and Required Theory

\[
\text{latency}_i = T_{receive,i} - T_{send,i}
\]

\[
\text{jitter}_i = |\text{latency}_i - \text{latency}_{i-1}|
\]

\[
\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i
\]

Percentiles are computed by sorting latency values and interpolating at the requested rank. P99 is important because the mean can hide rare delay spikes that still break a control deadline.

## In-Lab Method

1. Generate 60 packets.
2. Assign each packet a send time and simulated network delay.
3. Compute latency as receive time minus send time.
4. Compute jitter as the absolute difference between consecutive latency values.
5. Report mean, standard deviation, P95, P99, and average jitter.

In [1]:
import math
import random
import statistics

def percentile(values, pct):
    values = sorted(values)
    pos = (len(values) - 1) * pct
    lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo] * (hi - pos) + values[hi] * (pos - lo)

rng = random.Random(341402)
latencies = []
for packet_id in range(60):
    delay = 8.0 + rng.uniform(-1.4, 1.4) + rng.expovariate(1 / 1.8)
    if packet_id % 17 == 0:
        delay += rng.uniform(0, 4.0)
    latencies.append(delay)
jitters = [abs(latencies[i] - latencies[i - 1]) for i in range(1, len(latencies))]

print("EXP 2 - IN-LAB RESULT")
print(f"Packets         : {len(latencies)}")
print(f"Mean latency    : {statistics.mean(latencies):.2f} ms")
print(f"Std deviation   : {statistics.stdev(latencies):.2f} ms")
print(f"P95 latency     : {percentile(latencies, 0.95):.2f} ms")
print(f"P99 latency     : {percentile(latencies, 0.99):.2f} ms")
print(f"Average jitter  : {statistics.mean(jitters):.2f} ms")

EXP 2 - IN-LAB RESULT
Packets         : 60
Mean latency    : 10.16 ms
Std deviation   : 2.33 ms
P95 latency     : 14.35 ms
P99 latency     : 16.33 ms
Average jitter  : 2.56 ms


## Post-Lab Method

The post-lab cell injects 0-20 ms random delay to emulate congestion. It compares baseline and congested traffic, then evaluates a moving-average jitter buffer.

Trade-off:

- larger buffer window -> lower jitter
- larger buffer window -> more waiting delay

For control traffic, smoothing is useful only if the added delay remains within the control-loop deadline.

In [2]:
import math
import random
import statistics

def percentile(values, pct):
    values = sorted(values)
    pos = (len(values) - 1) * pct
    lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo] * (hi - pos) + values[hi] * (pos - lo)

def stats(extra_delay=False, seed=0):
    rng = random.Random(seed)
    lat = [8 + rng.uniform(-1.4, 1.4) + rng.expovariate(1 / 1.8) + (rng.uniform(0, 20) if extra_delay else 0) for _ in range(60)]
    jit = [abs(lat[i] - lat[i - 1]) for i in range(1, len(lat))]
    return statistics.mean(lat), statistics.stdev(lat), percentile(lat, 0.95), percentile(lat, 0.99), statistics.mean(jit)

print("EXP 2 - POST-LAB CONGESTION COMPARISON")
print(f"{'Condition':22} {'Mean':>8} {'Std':>8} {'P95':>8} {'P99':>8} {'Jitter':>8}")
for name, extra, seed in [("Baseline", False, 341402), ("0-20 ms delay", True, 341403)]:
    mean, stdev, p95, p99, jitter = stats(extra, seed)
    print(f"{name:22} {mean:8.2f} {stdev:8.2f} {p95:8.2f} {p99:8.2f} {jitter:8.2f}")

EXP 2 - POST-LAB CONGESTION COMPARISON
Condition                  Mean      Std      P95      P99   Jitter
Baseline                  10.03     2.29    13.77    16.00     2.20
0-20 ms delay             20.40     6.79    30.50    31.77     7.56


## What to Write in the Lab Record

- Include the baseline timing table.
- Include the congestion comparison table.
- Explain why P95/P99 are more meaningful than the mean for safety-critical messages.
- State the moving-average window that gives the best jitter-delay trade-off.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html